<a href="https://colab.research.google.com/github/christinatras/AAI2026/blob/dev/BUS4_118S_ML_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from google.colab import files

# DATASET LINK: https://www.kaggle.com/datasets/fratzcan/usa-house-prices?resource=download

# Load data from CSV
df = pd.read_csv("USA_Housing_Dataset.csv")

# Display column names
print("DataFrame Columns: ", df.columns.tolist())

# Features and target with updated columns
X = df[['sqft_living', 'location']]
y = df['price']

# Preprocessing: One-hot encode the location column
preprocessor = ColumnTransformer(
    transformers=[
        ('location', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['location'])
    ], remainder='passthrough')

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

# Train model
model.fit(X_train, y_train)

# Make prediction for a new house: 2000 sq ft in Suburb
new_house = pd.DataFrame({'sqft_living': [2000], 'location': ['Suburb']})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in Suburbs: ${predicted_price[0]:,.2f}")

# Display model coefficients
feature_names = (model.named_steps['preprocessor'] \
                 .named_transformers_['location'] \
                 .get_feature_names_out(['location'])).tolist() + \
                ['square_footage']
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

DataFrame Columns:  ['price', 'sqft_living', 'location']
Predicted price for a 2000 sq ft house in Suburbs: $535,435.31

Model Coefficients:
location_Downtown: 88.13
location_Rural: -18922.42
location_Suburb: 18834.28
square_footage: 267.48


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# DATASET LINK: https://www.kaggle.com/code/danishmubashar/telco-customer-churn-80-accuracy/input

# Load customer dataset
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Display colums to identify features in database
print("DataFrame Columns: ", df.columns.tolist())

# Features and target
X = df[['gender', 'SeniorCitizen', 'tenure', 'PhoneService',
'MultipleLines']]
y = df['Churn']

# Preprocessing: Scale numerical features and one-hot encode categorical features, Senior Citizen does not appear as specific numeric, rather 1 for senior and 0 not
preprocessor = ColumnTransformer(
transformers=[
('num', StandardScaler(), ['SeniorCitizen', 'tenure']),
('cat', OneHotEncoder(sparse_output=False), ['gender', 'PhoneService',
'MultipleLines'])
])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
('preprocessor', preprocessor),
('classifier', LogisticRegression(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
'gender': ['Female'],
'SeniorCitizen': [1], #In this dataset, 1 = senior, 0 = not
'tenure': [11], #How many months the customer has stayed with the company
'PhoneService': ['Yes'],
'MultipleLines': ['No']
})
churn_probability = model.predict_proba(new_customer)[0][1] # Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0
print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names = (model.named_steps['preprocessor']
.named_transformers_['cat']
.get_feature_names_out(['gender', 'PhoneService', 'MultipleLines'])).tolist() + ['SeniorCitizen', 'tenure', 'PhoneService', 'MultipleLines']
coefficients = model.named_steps['classifier'].coef_[0]
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
  print(f"{feature}: {coef:.2f}")

DataFrame Columns:  ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Churn Probability for new customer: 0.51
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
gender_Female: 0.32
gender_Male: -1.14
PhoneService_No: -0.25
PhoneService_Yes: -0.31
MultipleLines_No: -0.28
MultipleLines_No phone service: -0.27
MultipleLines_Yes: -0.60
SeniorCitizen: -0.28
tenure: 0.33


In [13]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# DATASET LINK: https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset/data

# Load dataset
df = pd.read_csv("/content/bank.csv")

# Define numerical and categorical features
numerical_features = ['age', 'balance']
categorical_features = ['job', 'marital', 'education']

# Create preprocessor using ColumnTransformer for numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# Apply preprocessing
X_processed = preprocessor.fit_transform(df)

# Determine optimal number of clusters using elbow method
inertia = []
K = range(1, 11) # Wider curve range
for k in K:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
  kmeans.fit(X_processed)
  inertia.append(kmeans.inertia_)


# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(K, inertia, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.savefig('elbow_plot.png')
plt.close()

# Apply K-Means with optimal K (e.g., 3 based on elbow method)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
df['cluster'] = kmeans.fit_predict(X_processed)

# Analyze numerical clusters
cluster_numeric_summary = df.groupby('cluster')[numerical_features].mean().round(2)
print("Cluster Numerical Characteristics:")
print(cluster_numeric_summary)

# Analyze categorical clusters
cluster_cat_summary = df.groupby('cluster')[categorical_features].agg(pd.Series.mode)
print("Cluster Categorial Characteristics:")
print(cluster_cat_summary)


# Targeted strategies
for cluster in range(optimal_k):
    avg_balance = cluster_numeric_summary.loc[cluster, 'balance']
    avg_age = cluster_numeric_summary.loc[cluster, 'age']

    print(f"\nCluster {cluster} Strategy:")
    if avg_balance > 1000:
        print("High-balance customers: Offer premium savings/investment products or VIP support.")
    elif avg_age < 30:
        print("Younger segment: Promote mobile-first banking, starter credit products, and education content.")
    else:
        print("General segment: Use personalized offers based on job/education and optimize communication channels.")

# Save cluster assignments to CSV
df.to_csv('customer_segments_data.csv', index=False)

Cluster Numerical Characteristics:
           age  balance
cluster                
0        34.72   882.80
1        35.12  1612.13
2        55.74  2362.20
Cluster Categorial Characteristics:
                 job  marital  education
cluster                                 
0        blue-collar  married  secondary
1         management   single   tertiary
2            retired  married  secondary

Cluster 0 Strategy:
General segment: Use personalized offers based on job/education and optimize communication channels.

Cluster 1 Strategy:
High-balance customers: Offer premium savings/investment products or VIP support.

Cluster 2 Strategy:
High-balance customers: Offer premium savings/investment products or VIP support.
